# Ground Truth Generation

Generate 5 synthetic user questions per chunk with `gpt-4o-mini`, keyed by `chunk_id`. Saved to `data/ground-truth-retrieval.csv`, used by `retrieval-eval.ipynb` (and later the LLM evaluation) as the ground-truth set for Hit Rate / MRR.

In [1]:
import pandas as pd

In [2]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

In [3]:
import sys
sys.path.append('..')
from coffee_assistant import retrieval

documents = retrieval.get_index().docs

## Generate questions for one chunk (sanity check)

Before running this over all 504 chunks, try the prompt on a single one.

In [4]:
prompt_template = """
You emulate a user of our coffee assistant application.
Formulate 5 questions this user might ask based on a provided coffee article excerpt.
Make the questions specific to this exercise.
The record should contain the answer to the questions, and the questions should
be complete and not too short. Use as fewer words as possible from the record. 

The record:

doc_id: {doc_id}
chunk_id: {chunk_id}
title: {title}
content: {content}

Provide the output in parsable JSON without using code blocks:

{{"questions": ["question1", "question2", ..., "question5"]}}
""".strip()

In [5]:
prompt = prompt_template.format(**documents[0])

In [6]:
print(prompt)

You emulate a user of our coffee assistant application.
Formulate 5 questions this user might ask based on a provided coffee article excerpt.
Make the questions specific to this exercise.
The record should contain the answer to the questions, and the questions should
be complete and not too short. Use as fewer words as possible from the record. 

The record:

doc_id: 604727
chunk_id: 604727_0
title: Coffee
content: Coffee is a beverage brewed from roasted, ground coffee beans. Dark-colored and bitter, coffee has a stimulating effect on humans due to its caffeine content; however decaffeinated coffee is also commercially available. There are also various coffee substitutes.
Coffee production begins when the seeds from coffee cherries, the Coffea plant's fruits, are separated to produce unroasted green coffee beans. The "beans" are roasted and then ground into fine particles. Coffee is brewed from the ground roasted beans, which are typically steeped in hot water before being filtered ou

In [7]:
def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [8]:
questions = llm(prompt)

In [9]:
import json

json.loads(questions)

{'questions': ['What is the main ingredient used to brew coffee as mentioned in the article?',
  'What types of coffee beans are most commonly grown worldwide?',
  'How did the popularity of coffee spread according to the historical context in the excerpt?',
  'What impact does the coffee industry have on coffee farmers, based on the information provided?',
  'Can you list some common ways coffee is prepared and served as mentioned in the document?']}

In [10]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response

## Generate questions for all chunks

In [11]:
from tqdm.auto import tqdm

In [12]:
results = {}

In [13]:
for doc in tqdm(documents): 
    chunk_id = doc['chunk_id']
    if chunk_id in results:
        continue

    questions_raw = generate_questions(doc)
    questions = json.loads(questions_raw)
    results[chunk_id] = questions['questions']

  0%|          | 0/504 [00:00<?, ?it/s]

In [14]:
final_results = []

for chunk_id, questions in results.items():
    for q in questions:
        final_results.append((chunk_id, q))

In [15]:
final_results[0]

('604727_0',
 'What is the origin of coffee consumption according to the article?')

## Save ground truth

In [16]:
df_results = pd.DataFrame(final_results, columns=['chunk_id', 'question'])

In [17]:
df_results.to_csv('../data/ground-truth-retrieval.csv', index=False)